<p>
<font size='5' face='Georgia, Arial'>IIC2233 Apunte Programación Avanzada</font><br>
<font size='1'> Modificado en 2019-1 al 2026-2 por Equipo Docente IIC2233. </font>
</p>

# Tabla de contenidos

1. [Serialización de objetos](#serializacion)
    1. [JSON](#json)
        1. [Serialización de objetos personalizados](#objetos-personalizados)
            1. [Personalizar la serialización](#serializacion-personalizada)
            2. [Personalizar la deserialización](#deserializacion-personalizada)
        2. [Errores al deserializar JSON](#errores-json)

<a id="serializacion"></a>
# Serialización de objetos

Toda la información que almacena o transmite un computador termina representada mediante *bits* y *bytes*. Sin embargo, los programas trabajan con estructuras de mayor nivel, como listas, diccionarios e instancias de clases.

La **serialización** transforma el estado de un objeto a una representación que puede guardarse o transmitirse. Esa representación puede ser texto —como JSON— o una secuencia binaria, según el formato utilizado. Para enviar texto por una red, además es necesario **codificarlo** a bytes, por ejemplo con UTF-8.

Serializar un objeto no lo vuelve persistente por sí solo. La persistencia se obtiene cuando la representación se guarda en un archivo, base de datos u otro medio de almacenamiento.

<a id="json"></a>
## JSON

**JSON** (*JavaScript Object Notation*) es un formato de texto estándar para intercambiar datos entre programas. Es independiente del lenguaje de programación y resulta relativamente fácil de leer para una persona.

**Su sintaxis se parece a la de los diccionarios y listas de Python, pero no es idéntica.**

El módulo `json` transforma por defecto los siguientes tipos:

| Python | JSON |
|---|---|
| `dict` | objeto |
| `list`, `tuple` | arreglo |
| `str` | string |
| `int`, `float` | número |
| `True`, `False` | `true`, `false` |
| `None` | `null` |

Una tupla puede serializarse, pero al deserializarla se obtiene una lista, porque JSON no tiene un tipo tupla. Por defecto no se pueden serializar funciones, conjuntos ni instancias de clases personalizadas.

- `dumps` serializa un objeto y retorna un `str`.
- `loads` deserializa un `str` con JSON.
- `dump` serializa hacia un archivo de texto ya abierto.
- `load` deserializa desde un archivo ya abierto.

Consulta la [documentación oficial](https://docs.python.org/3.12/library/json.html#encoders-and-decoders).

Primero, veamos cómo se comporta JSON al serializar y deserializar una lista con distintos valores:

In [1]:
import json


lista = ["a", 1, 3, "hola"]
lista_serializada = json.dumps(lista)

print(f"Resultado serialización: {lista_serializada}")
print(f"Tipo de versión serializada: {type(lista_serializada)}")

lista_deserializada = json.loads(lista_serializada)
print(f"Resultado deserialización: {lista_deserializada}")

print()
print(f"Lista original: {lista}")
print(f"Lista deserializada: {lista_deserializada}")
print(f"¿Las listas son iguales? {lista == lista_deserializada}")
print(f"¿Las listas son el mismo objeto? {lista is lista_deserializada}")


Resultado serialización: ["a", 1, 3, "hola"]
Tipo de versión serializada: <class 'str'>
Resultado deserialización: ['a', 1, 3, 'hola']

Lista original: ['a', 1, 3, 'hola']
Lista deserializada: ['a', 1, 3, 'hola']
¿Las listas son iguales? True
¿Las listas son el mismo objeto? False


Pero hay que tener ojo con la transformación de algunos tipos de datos, ya que -como se explica en la [tabla de transformación](https://docs.python.org/3.12/library/json.html#encoders-and-decoders)- algunos objetos de Python comparten un mismo tipo de dato en el formato JSON: 

In [2]:
import json


tupla = (None, 1, 22.33, {"a": True, "b": False})
tupla_serializada = json.dumps(tupla)

print(f"Resultado serialización: {tupla_serializada}")
print(f"Tipo de versión serializada: {type(tupla_serializada)}")

tupla_deserializada = json.loads(tupla_serializada)
print(f"Resultado deserialización: {tupla_deserializada}")
print(f"Tipo de versión deserializada: {type(tupla_deserializada)}")

print()
print(f"Tupla original: {tupla}")
print(f"Tupla deserializada: {tupla_deserializada}")
print(f"¿Las tuplas son iguales? {tupla == tupla_deserializada}")
print(f"¿Las tuplas son el mismo objeto? {tupla is tupla_deserializada}")

Resultado serialización: [null, 1, 22.33, {"a": true, "b": false}]
Tipo de versión serializada: <class 'str'>
Resultado deserialización: [None, 1, 22.33, {'a': True, 'b': False}]
Tipo de versión deserializada: <class 'list'>

Tupla original: (None, 1, 22.33, {'a': True, 'b': False})
Tupla deserializada: [None, 1, 22.33, {'a': True, 'b': False}]
¿Las tuplas son iguales? False
¿Las tuplas son el mismo objeto? False


Las funciones `dump` y `load` trabajan con archivos o flujos ya abiertos:

- `dump(objeto, archivo)` escribe la representación JSON.
- `load(archivo)` lee un documento JSON y retorna su representación en Python.

Como JSON utiliza texto Unicode, es recomendable abrir los archivos explícitamente con codificación UTF-8.

In [3]:
import json
from os import path


lista = [1, 2, 3, 7, 8, 3]

with open(path.join("data", "mi_lista.json"), "w", encoding="utf-8") as archivo:
    json.dump(lista, archivo)

with open(path.join("data", "mi_lista.json"), "r", encoding="utf-8") as archivo:
    lista_cargada = json.load(archivo)

print(f"Lista original: {lista}")
print(f"Lista cargada : {lista_cargada}")
print(f"¿Las listas son iguales? {lista == lista_cargada}")
print(f"¿Las listas son el mismo objeto? {lista is lista_cargada}")

Lista original: [1, 2, 3, 7, 8, 3]
Lista cargada : [1, 2, 3, 7, 8, 3]
¿Las listas son iguales? True
¿Las listas son el mismo objeto? False


<a id="objetos-personalizados"></a>
### Serialización de objetos personalizados

Ahora que entendemos mejor la serialización y deserialización, puede surgir la siguiente duda:

> ¿Qué ocurre si la información se encuentra en instancias de clases personalizadas?

Para serializar una instancia personalizada debemos transformarla a una estructura compatible con JSON, como un diccionario.

In [4]:
import json


class Mascota:

    def __init__(self, nombre: str, edad: int, especie: str, id_: int) -> None:
        self.nombre = nombre
        self.edad = edad
        self.especie = especie
        self.id_ = id_

In [5]:
m = Mascota("Luna", 8, "Gato", 0)

json_string = json.dumps(m.__dict__)
print("Datos en formato JSON:", type(json_string), json_string)

json_deserializado = json.loads(json_string)
print("Datos en formato Python:", type(json_deserializado), json_deserializado)

Datos en formato JSON: <class 'str'> {"nombre": "Luna", "edad": 8, "especie": "Gato", "id_": 0}
Datos en formato Python: <class 'dict'> {'nombre': 'Luna', 'edad': 8, 'especie': 'Gato', 'id_': 0}


Transformar `m.__dict__` permite serializar los atributos, pero al deserializar obtenemos un diccionario, no una nueva instancia de `Mascota`.

Para personalizar ambos procesos utilizaremos una subclase de `json.JSONEncoder` y una función entregada mediante `object_hook`.

<a id="serializacion-personalizada"></a>
#### Personalizar la serialización en `json`

`JSONEncoder` puede extenderse sobrescribiendo `default(obj)`. Si reconocemos el tipo, retornamos una estructura compatible con JSON. Si no lo reconocemos, llamamos `super().default(obj)` para que se levante `TypeError`.

No se llama al método de la superclase con el diccionario ya transformado; basta con retornarlo.

In [6]:
import json
from datetime import datetime


class MascotaEncoder(json.JSONEncoder):

    def default(self, obj):
        if isinstance(obj, Mascota):
            return {
                "tipo": "Mascota",
                "Mascota_id": obj.id_,
                "nombre": obj.nombre,
                "edad": obj.edad,
                "especie": obj.especie,
                # Aproximación: el año exacto depende del cumpleaños.
                "ano_nacimiento_aproximado": datetime.now().year - obj.edad,
            }
        return super().default(obj)


m1 = Mascota("Pepa", 70, "Tortuga", 1)
m2 = Mascota("Cachirulo", 5, "Gato", 2)
m3 = Mascota("Guillermo", 0, "Gato", 3)

Probamos la serialización usando el método por defecto:

In [7]:
json_string = json.dumps(m1.__dict__)

print(json_string)

{"nombre": "Pepa", "edad": 70, "especie": "Tortuga", "id_": 1}


Ahora, comparemos el resultado al serializar usando nuestro método personalizado:

In [8]:
json_string = json.dumps(m1, cls=MascotaEncoder)
print(json_string)

json_string = json.dumps(m2, cls=MascotaEncoder)
print(json_string)

json_string = json.dumps(m3, cls=MascotaEncoder)
print(json_string)


{"tipo": "Mascota", "Mascota_id": 1, "nombre": "Pepa", "edad": 70, "especie": "Tortuga", "ano_nacimiento_aproximado": 1956}
{"tipo": "Mascota", "Mascota_id": 2, "nombre": "Cachirulo", "edad": 5, "especie": "Gato", "ano_nacimiento_aproximado": 2021}
{"tipo": "Mascota", "Mascota_id": 3, "nombre": "Guillermo", "edad": 0, "especie": "Gato", "ano_nacimiento_aproximado": 2026}


También podemos extender el mismo `JSONEncoder` para que maneje **más de una clase personalizada**. Basta con agregar más ramas `isinstance` dentro del método `default`. Si el objeto no corresponde a ningún tipo conocido, la llamada a `super().default(obj)` levantará un `TypeError`, avisándonos que ese tipo no es serializable.

In [9]:
class Veterinario:
    def __init__(self, nombre: str, especialidad: str) -> None:
        self.nombre = nombre
        self.especialidad = especialidad


class MascotaVeterinarioEncoder(json.JSONEncoder):

    def default(self, obj):
        if isinstance(obj, Mascota):
            return {
                "tipo": "Mascota",
                "nombre": obj.nombre,
                "edad": obj.edad,
                "especie": obj.especie,
            }
        if isinstance(obj, Veterinario):
            return {
                "tipo": "Veterinario",
                "nombre": obj.nombre,
                "especialidad": obj.especialidad,
            }
        # Para cualquier otro tipo no soportado, levanta TypeError
        return super().default(obj)


v = Veterinario("Dra. Torres", "Exóticos")

# Serializar un Veterinario
print("Veterinario:")
print(json.dumps(v, cls=MascotaVeterinarioEncoder))

# Serializar una Mascota con el mismo encoder
print("\nMascota:")
print(json.dumps(m1, cls=MascotaVeterinarioEncoder))

# Serializar una lista mixta
print("\nLista mixta:")
print(json.dumps([m2, v, m3], cls=MascotaVeterinarioEncoder, indent=2))

# Intentar serializar un tipo no soportado → TypeError
try:
    json.dumps(object(), cls=MascotaVeterinarioEncoder)
except TypeError as e:
    print(f"\nTipo no soportado → {e}")

Veterinario:
{"tipo": "Veterinario", "nombre": "Dra. Torres", "especialidad": "Ex\u00f3ticos"}

Mascota:
{"tipo": "Mascota", "nombre": "Pepa", "edad": 70, "especie": "Tortuga"}

Lista mixta:
[
  {
    "tipo": "Mascota",
    "nombre": "Cachirulo",
    "edad": 5,
    "especie": "Gato"
  },
  {
    "tipo": "Veterinario",
    "nombre": "Dra. Torres",
    "especialidad": "Ex\u00f3ticos"
  },
  {
    "tipo": "Mascota",
    "nombre": "Guillermo",
    "edad": 0,
    "especie": "Gato"
  }
]

Tipo no soportado → Object of type object is not JSON serializable


<a id="deserializacion-personalizada"></a>
#### Personalizar la deserialización en `json`

`object_hook` recibe el resultado de **cada objeto JSON** (`{...}`) convertido inicialmente a `dict`. No se ejecuta para arreglos, strings, números, booleanos ni `null`.

La función debe reconocer solo los diccionarios que le corresponden y retornar sin cambios los demás.

Por ejemplo, si queremos cargar datos JSON en una lista de tuplas en vez de un diccionario:

In [10]:
def funcion_lista_hook(diccionario: dict):
    return [(key, value) for key, value in diccionario.items()]


json_string = '{"nombre": "Luna", "edad": 8, "especie": "Gato", "ano_nacimiento": 2017}'
datos = json.loads(json_string, object_hook=funcion_lista_hook)

print(datos)

[('nombre', 'Luna'), ('edad', 8), ('especie', 'Gato'), ('ano_nacimiento', 2017)]


Para reconstruir mascotas agregamos un discriminador, `"tipo": "Mascota"`. Así el *hook* puede distinguir una mascota de cualquier otro diccionario, incluso dentro de objetos anidados.

In [11]:
def funcion_mascota_hook(diccionario: dict):
    if diccionario.get("tipo") != "Mascota":
        return diccionario
    return Mascota(
        nombre=diccionario["nombre"],
        edad=diccionario["edad"],
        especie=diccionario["especie"],
        id_=diccionario["Mascota_id"],
    )


lista_json_mascotas = [
    json.dumps(mascota, cls=MascotaEncoder)
    for mascota in (m, m1, m2, m3)
]

lista_objetos_mascotas = [
    json.loads(texto, object_hook=funcion_mascota_hook)
    for texto in lista_json_mascotas
]

for mascota in lista_objetos_mascotas:
    print(mascota.id_, mascota.nombre, mascota.especie)

0 Luna Gato
1 Pepa Tortuga
2 Cachirulo Gato
3 Guillermo Gato


`object_hook` se aplica a cada **objeto JSON**, comenzando por los objetos más internos. El resultado de una llamada puede formar parte del diccionario que reciba una llamada posterior:

In [12]:
def funcion_lista_hook(diccionario):
    """Esta función transforma un diccionario en un número"""
    print(f"Me llegó el diccionario: {diccionario}")
    valor = 33 if len(diccionario) > 1 else 42
    print(f"Lo transformaré en {valor}\n")
    return valor


json_string = '{"a": {"b": 1, "c": [2, 3, {}], "d": null, "e": {"f": true}}}'
datos = json.loads(json_string, object_hook=funcion_lista_hook)

print(datos)


Me llegó el diccionario: {}
Lo transformaré en 42

Me llegó el diccionario: {'f': True}
Lo transformaré en 42

Me llegó el diccionario: {'b': 1, 'c': [2, 3, 42], 'd': None, 'e': 42}
Lo transformaré en 33

Me llegó el diccionario: {'a': 33}
Lo transformaré en 42

42


El proceso ocurre de adentro hacia afuera y el resultado de cada llamada se incorpora al nivel que la contiene.

Los arreglos y valores primitivos no pasan directamente por `object_hook`; solo lo hacen los objetos JSON convertidos inicialmente a diccionarios.

<a id="errores-json"></a>
### Errores al deserializar JSON

Si el texto no cumple la sintaxis JSON, `loads` y `load` levantan `json.JSONDecodeError`. Podemos capturar la excepción para informar el problema sin detener toda la ejecución.

In [13]:
texto_json = '["Arreglo que no es cerrado"'
try:
    json.loads(texto_json)
except json.JSONDecodeError as error:
    print(f"JSON inválido: {error}")

JSON inválido: Expecting ',' delimiter: line 1 column 29 (char 28)


In [14]:
texto_json = '{"mascota_mas_vieja": "Pepa", "mascota_mas_nueva" "Guillermo"}'
try:
    json.loads(texto_json)
except json.JSONDecodeError as error:
    print(f"JSON inválido: {error}")

JSON inválido: Expecting ':' delimiter: line 1 column 51 (char 50)


In [15]:
texto_json = '["Luna", "Cachirulo" "Guillermo"]'
try:
    json.loads(texto_json)
except json.JSONDecodeError as error:
    print(f"JSON inválido: {error}")

JSON inválido: Expecting ',' delimiter: line 1 column 22 (char 21)


En los tres casos se identifica un `JSONDecodeError` y el programa continúa. El mensaje incluye la línea, columna y posición aproximada donde se detectó la sintaxis inválida.